In [166]:
from sklearn.ensemble import RandomForestRegressor
from greedy_boruta  import GreedyBorutaPy

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"]=(20,10)
import seaborn as sns;sns.set()
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
from datetime import datetime,time
from plotly.offline import iplot,plot,init_notebook_mode,download_plotlyjs
%matplotlib inline 
init_notebook_mode(connected=True)
import math
import import_ipynb
import missingno as msno
import os
from xgboost import XGBRegressor

In [51]:
old = 'c:\\Users\\Omar\\Desktop\\Omar_Files\\Python_Analysis\\Pepper_Prices_Analysis\\Pepper_Prices_Analysis'
os.chdir(old)
os.getcwd() 
import os
#%pwd
#os.chdir("../")
%pwd

'c:\\Users\\Omar\\Desktop\\Omar_Files\\Python_Analysis\\Pepper_Prices_Analysis\\Pepper_Prices_Analysis'

In [210]:
df=pd.read_csv('Data_Sets/actual_data.csv')
df=df[df["total_volume"]!=-1]
df.interpolate(method="linear",inplace=True,limit_direction="backward")

df['week_start_dt']=pd.to_datetime(df['week_start_dt'])
df['week_end_dt']=pd.to_datetime(df['week_end_dt'])


numerical_cols=df.select_dtypes(include=np.number).columns.tolist()
categorical_cols=df.select_dtypes(include='object').columns.tolist()
boolean_cols=df.select_dtypes(include='bool').columns.tolist()
date_cols=df.select_dtypes(include='datetime').columns.tolist()

print(f"numerical_cols : \n{numerical_cols}")
print("--------------------------------------------------")
print(f"categorical_cols : \n{categorical_cols}")
print("--------------------------------------------------")
print(f"boolean_cols : \n{boolean_cols}")
print("--------------------------------------------------")
print(f"date_cols : \n{date_cols}")

numerical_cols : 
['vietnam_season', 'price', 'total_volume', 'brazil', 'india', 'vietnam', 'indonesia', 'china', 'jordan_max_price', 'jordan_min_price', 'demand', 'supply']
--------------------------------------------------
categorical_cols : 
['p_color']
--------------------------------------------------
boolean_cols : 
['brazil_season', 'indonesia_season', 'india_season', 'china_season']
--------------------------------------------------
date_cols : 
['week_start_dt', 'week_end_dt']


In [212]:
X = df.drop(columns=["price", "week_start_dt", "week_end_dt"],axis=1)
y = df["price"]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dummy_na=True)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

selector = GreedyBorutaPy(
    rf,
    n_estimators='auto',   # or 'split'
    verbose=1,
    random_state=42
)

selector.fit(X.values, y.values)

# Results
confirmed_features = X.columns[selector.support_].tolist()
tentative_features = X.columns[selector.support_weak_].tolist() if hasattr(selector, 'support_weak_') else []

print("Confirmed important:", confirmed_features)
print("Tentative:", tentative_features)
print("Rejected:", X.columns[~selector.support_].tolist())

Iteration: 1 / 9
Iteration: 2 / 9
Iteration: 3 / 9
Iteration: 4 / 9
Iteration: 5 / 9
Iteration: 6 / 9
Iteration: 7 / 9
Iteration: 8 / 9


GreedyBorutaPy finished running.

Iteration: 	9 / 9
Confirmed: 	7
Tentative: 	0
Rejected: 	11
Confirmed important: ['total_volume', 'brazil', 'vietnam', 'indonesia', 'jordan_max_price', 'jordan_min_price', 'demand']
Tentative: []
Rejected: ['vietnam_season', 'india', 'china', 'brazil_season', 'indonesia_season', 'india_season', 'china_season', 'supply', 'p_color_red', 'p_color_yellow', 'p_color_nan']


In [213]:
selected_features_1=pd.concat([X[confirmed_features].astype("int"),y],axis=1)
selected_features_1

,total_volume,brazil,vietnam,indonesia,jordan_max_price,jordan_min_price,demand,price
57,1596040,10793,1519588,0,6,6,0,6.599075
58,1596040,10793,1519588,0,7,7,0,7.175335
59,1596040,10793,1519588,0,7,7,16,7.300575
60,2295578,5677,2274625,0,7,7,271,7.379675
61,2295578,5677,2274625,0,7,7,42,7.175335
...,...,...,...,...,...,...,...,...
1267,2761128,8695,2530027,180220,7,7,88,7.334644
1268,2761128,8695,2530027,180220,9,9,305,9.008137
1269,2665343,167,2521054,78334,7,7,97,7.259712
1270,2665343,167,2521054,78334,7,6,102,6.910027
